# Avg-Latent Evaluation — 4 Experiments (N=2000)

Short version of `eval_all_experiments.ipynb`, scoped to just the
latent-averaging baseline: for each of SkewedTarget / BalancedTarget /
GenderInterpolation / AgeInterpolation, compares `source` (unguided) vs
`avg_latent`.

`avg_latent` is built in this notebook (see `build_avg_latent_scribble` in
the Helpers cell): generate each target group's portrait as a latent (no
decode), weighted-average the latents, decode once, then run HED on that
single image — mirrors `build_avg_scribble_latent` in
`scripts/eval_baselines.py`, but self-contained so no separate SLURM job is
needed first. The result is cached to
`experiments/<Experiment>/scribble_avg_latent.png` so re-running the
notebook doesn't rebuild it.


In [1]:
import os, sys, getpass
from pathlib import Path

if 'google.colab' in str(get_ipython()):
    from google.colab import userdata

    !pip install -q controlnet_aux diffusers transformers accelerate xformers scikit-learn matplotlib Pillow pandas

    github_token = userdata.get('GITHUB')  # Colab secret named "GITHUB", or paste when prompted
    if not github_token:
        github_token = getpass.getpass("Enter your GitHub personal access token: ")

    repo_url  = f"https://{github_token}@github.com/orineo1/conditional-matching-paper.git"
    repo_name = "conditional-matching-paper"
    branch    = "claude/latent-avg-baseline"   # or "main" once the PR is merged

    if not os.path.exists(repo_name):
        !git clone {repo_url}
    else:
        print(f"✅ Repo '{repo_name}' already cloned — pulling latest...")
        !cd {repo_name} && git pull

    !cd {repo_name} && git checkout {branch}

    repo_path = f"/content/{repo_name}/SD_cond_SD_controlnet"
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

    print(f"\n✅ Repo ready. Branch: {branch}")
    print(f"📁 Python path: {repo_path}")

# ── sys.path: add src/ ─────────────────────────────────────────────────────
if 'repo_path' in globals():
    _REPO = Path(repo_path)          # set by the Colab git-clone cell above
else:
    _NB_DIR = Path().resolve()
    _REPO   = _NB_DIR.parent if (_NB_DIR.parent / 'src').exists() else _NB_DIR

_SRC_DIR = str(_REPO / 'src')
if _SRC_DIR not in sys.path:
    sys.path.insert(0, _SRC_DIR)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 115.1 MB/s eta 0:00:00
Cloning into 'conditional-matching-paper'...
remote: Enumerating objects: 11691, done.
remote: Counting objects: 100% (410/410), done.
remote: Compressing objects: 100% (159/159), done.
remote: Total 11691 (delta 324), reused 299 (delta 251), pack-reused 11281 (from 3)
Receiving objects: 100% (11691/11691), 1.21 GiB | 11.36 MiB/s, done.
Resolving deltas: 100% (5220/5220), done.
branch 'claude/latent-avg-baseline' set up to track 'origin/claude/latent-avg-baseline'.
Switched to a new branch 'claude/latent-avg-baseline'

✅ Repo ready. Branch: claude/latent-avg-baseline
📁 Python path: /content/conditional-matching-paper/SD_cond_SD_controlnet


In [2]:
import os, sys, json
import numpy as np
import torch
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from pathlib import Path
from IPython.display import display
from tqdm.notebook import tqdm

# ── sys.path: add src/ relative to this notebook ─────────────────────────────
_NB_DIR  = Path().resolve()
_REPO    = _NB_DIR.parent if (_NB_DIR.parent / 'src').exists() else _NB_DIR
_SRC_DIR = str(_REPO / 'src')
if _SRC_DIR not in sys.path:
    sys.path.insert(0, _SRC_DIR)

from models      import load_models
from clip_utils  import load_clip_model, encode_images_clip
from metrics     import compute_mmd, compute_swd
from image_utils import latent_to_pil
from scripts.run_mlgd_f import compute_clip_softmax  # noqa: used in eval loop

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cuda


In [3]:
## 2. Config

REPO_EXPERIMENTS = str(_REPO / 'experiments')
N_EVAL           = 2000
N_PREVIEW        = 5

SEED             = 42   # target generation + neutral-eval generation seed
SWD_SEED         = 123  # fixed seed for compute_swd's random projections --
                         # reset before EVERY compute_swd call (see helpers)
                         # so all methods within an experiment are scored
                         # against identical projections, and the whole
                         # notebook is reproducible run to run.

CONTROLNET_SCALE = 0.5
NEUTRAL_PROMPT   = 'a superrealistic professional photograph of'
MAN_PROMPT       = 'a superrealistic portrait photograph of a man, studio lighting'
WOMAN_PROMPT     = 'a superrealistic portrait photograph of a woman, studio lighting'

# Same 4 experiment configs as eval_all_experiments.ipynb (target groups/prompts
# must match what MLGD-F/the avg-latent baseline were themselves evaluated against).
EXPERIMENT_CONFIGS = {
    'SkewedTarget': dict(
        mode   = 'binary',
        groups = [
            dict(label='Man',   prompt=MAN_PROMPT,   frac=0.25, color='royalblue'),
            dict(label='Woman', prompt=WOMAN_PROMPT, frac=0.75, color='crimson'),
        ],
    ),
    'BalancedTarget': dict(
        mode   = 'binary',
        groups = [
            dict(label='Man',   prompt=MAN_PROMPT,   frac=0.5, color='royalblue'),
            dict(label='Woman', prompt=WOMAN_PROMPT, frac=0.5, color='crimson'),
        ],
    ),
    'GenderInterpolation': dict(
        mode   = 'multiclass',
        groups = [
            dict(label='Woman',                  prompt='superrealistic portrait photograph of a woman, extremely feminine features, studio lighting',                                                    frac=0.25, color='crimson'),
            dict(label='Woman w/ masc features', prompt='a superrealistic portrait photograph of a woman with masculine features, heavy brow ridge, studio lighting',                                    frac=0.25, color='orchid'),
            dict(label='Man w/ fem features',    prompt='a superrealistic portrait photograph of a man with extremely feminine features, soft delicate face, high cheekbones, studio lighting',         frac=0.25, color='slategray'),
            dict(label='Man',                    prompt='a superrealistic portrait photograph of a man, extremely masculine features, studio lighting',                                                  frac=0.25, color='steelblue'),
        ],
    ),
    'AgeInterpolation': dict(
        mode     = 'age',
        age_min  = 40,
        age_max  = 79,
        age_step = 1,
    ),
}
EXPERIMENT_CONFIGS = {
    k: v for k, v in EXPERIMENT_CONFIGS.items()
    if k in ('SkewedTarget', 'BalancedTarget')
}



METHOD_NAMES  = ['source', 'avg_latent']
METHOD_COLORS = {'source': '#888888', 'avg_latent': '#2ca02c'}
CONTROLNET_SCALE      = 0.5   # target-building (matches training's --controlnet_scale)
EVAL_CONTROLNET_SCALE = 0.8   # final per-method generation (matches evaluate_distribution_mmd's hardcoded loop-time value)

print('Config OK')


Config OK


In [4]:
## 3. Load Models
from controlnet_aux import HEDdetector

architect, sprinter        = load_models(device)
clip_model, clip_processor = load_clip_model(device)
hed                        = HEDdetector.from_pretrained('lllyasviel/Annotators')
print('Models loaded.')


/usr/local/lib/python3.13/dist-packages/controlnet_aux/mediapipe_face/mediapipe_face_common.py:7: UserWarning: The module 'mediapipe' is not installed. The package will have limited functionality. Please install it using the command: pip install 'mediapipe'
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.13/dist-packages/controlnet_aux/segment_anything/modeling/tiny_vit_sam.py:654: UserWarning: Overwriting tiny_vit_5m_224 in registry with controlnet_aux.segmen

config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors: reconstructing file:   0%|          |  0.00B / 2.50GB            

diffusion_pytorch_model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


...

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 14.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 5.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 13.99 GiB is allocated by PyTorch, and 477.20 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
## 4. Helpers

def gen_images(scribble_pil, prompt, n, seed=None, controlnet_scale=CONTROLNET_SCALE):
    sprinter.vae.to(dtype=torch.float16)
    generator = torch.Generator(device=sprinter.device).manual_seed(seed) if seed is not None else None
    imgs = []
    with torch.no_grad():
        for start in range(0, n, 2):
            bs = min(2, n - start)
            imgs.extend(sprinter(
                prompt=[prompt]*bs, image=[scribble_pil]*bs,
                num_inference_steps=2, guidance_scale=0.0,
                controlnet_conditioning_scale=controlnet_scale,
                output_type='pil', generator=generator,
            ).images)
    sprinter.vae.to(dtype=torch.float32)
    return imgs

def clip_embed(images):
    tensors = torch.cat([TF.to_tensor(img).unsqueeze(0) for img in images]).to(device)
    clip_model.to(device)
    with torch.no_grad():
        embs = encode_images_clip(tensors, clip_model, clip_processor)
    clip_model.to('cpu')
    return embs

def compute_swd_seeded(x, y, seed=SWD_SEED):
    """compute_swd draws random projection directions via torch.randn with no
    seeding of its own -- reseed right before every call so results are both
    reproducible run-to-run and comparable method-to-method (same projections
    used for source/lgd_cm/pgd within one experiment)."""
    torch.manual_seed(seed)
    return compute_swd(x, y)

def binomial_ci(n_success, n_total, z=1.96):
    p  = n_success / n_total
    se = np.sqrt(p * (1-p) / n_total)
    return p, p - z*se, p + z*se

def show_image_row(images, title, n=N_PREVIEW):
    n_show = min(n, len(images))
    fig, axes = plt.subplots(1, n_show, figsize=(3*n_show, 3))
    if n_show == 1: axes = [axes]
    for ax, img in zip(axes, images[:n_show]):
        ax.imshow(img); ax.axis('off')
    fig.suptitle(title, fontweight='bold')
    plt.tight_layout(); display(fig); plt.close()

def build_avg_latent_scribble(cfg, source_scribble, seed=SEED, cn_scale=CONTROLNET_SCALE):
    """Generate each target group's portrait as a VAE latent (no decode),
    weighted-average the latents, decode once, then run HED -- avoids the
    pixel-space ghosting of averaging each group's separately-decoded HED
    scribble (see build_avg_scribble_latent in scripts/eval_baselines.py)."""
    if cfg['mode'] == 'age':
        ages    = list(range(cfg['age_min'], cfg['age_max'] + 1, cfg['age_step']))
        prompts = [f'a superrealistic portrait photograph of a {age}-year-old man, '
                   'studio lighting, sharp focus, photographic' for age in ages]
        fracs   = [1.0 / len(ages)] * len(ages)
    else:
        prompts = [g['prompt'] for g in cfg['groups']]
        fracs   = [g['frac']   for g in cfg['groups']]

    sprinter.vae.to(dtype=torch.float16)
    avg_latent = None
    for i, (prompt, frac) in enumerate(zip(prompts, fracs)):
        generator = torch.Generator(device=sprinter.device).manual_seed(seed + i)
        with torch.no_grad():
            latent = sprinter(
                prompt=[prompt], image=[source_scribble],
                num_inference_steps=2, guidance_scale=0.0,
                controlnet_conditioning_scale=cn_scale,
                output_type='latent', generator=generator,
            ).images
        avg_latent = frac * latent if avg_latent is None else avg_latent + frac * latent

    # SDXL's VAE overflows to NaN when decoded in fp16 -- the pipeline normally
    # upcasts to fp32 before its own internal decode, but output_type='latent'
    # skips that decode entirely, so we upcast by hand before calling it here.
    sprinter.vae.to(dtype=torch.float32)
    with torch.no_grad():
        portrait = latent_to_pil(avg_latent.float(), sprinter.vae, sprinter.image_processor)

    scribble_np = np.array(hed(portrait, scribble=True)).astype(np.uint8)
    return Image.fromarray(scribble_np)

def load_scribble(base_dir, name):
    p = Path(base_dir) / SCRIBBLE_FILES[name]
    if p.exists():
        return Image.open(p)
    raise FileNotFoundError(f'Scribble not found: {p}')

print('Helpers ready.')

In [ ]:
REPO_EXPERIMENTS = '/content/conditional-matching-paper/SD_cond_SD_controlnet/experiments'

In [ ]:
## 5. Main Evaluation Loop

all_results = {}

for exp_name, cfg in EXPERIMENT_CONFIGS.items():
    print(f'\n{"="*60}')
    print(f'  EXPERIMENT: {exp_name}')
    print(f'{"="*60}')

    base_dir  = Path(REPO_EXPERIMENTS) / exp_name

    scribbles = {}
    for name in METHOD_NAMES:
        if name == 'avg_latent':
            avg_latent_path = base_dir / SCRIBBLE_FILES['avg_latent']
            if avg_latent_path.exists():
                scribbles[name] = Image.open(avg_latent_path)
                print(f'  Loaded cached avg_latent scribble from {avg_latent_path}')
            else:
                print('  Building avg_latent scribble (averaging VAE latents, one decode)...')
                scribbles[name] = build_avg_latent_scribble(cfg, scribbles['source'])
                scribbles[name].save(avg_latent_path)
                print(f'  Saved {avg_latent_path}')
        else:
            scribbles[name] = load_scribble(base_dir, name)

    n = len(scribbles)
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    if n == 1: axes = [axes]
    for ax, (name, img) in zip(axes, scribbles.items()):
        ax.imshow(img, cmap='gray'); ax.set_title(name); ax.axis('off')
    plt.suptitle(f'{exp_name} — Scribbles', fontweight='bold')
    plt.tight_layout(); display(fig); plt.close()

    # ── Plot avg_latent alone, no title/axes -- identical to the raw PNG saved by eval_baselines.py ──
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(scribbles['avg_latent'], cmap='gray')
    ax.axis('off')
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    display(fig); plt.close()

    source_scribble = scribbles['source']
    mode = cfg['mode']

    # ── Build target distribution (N=2000) ──
    print(f'\nBuilding target distribution (N={N_EVAL})...')
    target_imgs_by_group = {}

    if mode in ('binary', 'multiclass'):
        all_target_imgs = []
        for i, g in enumerate(cfg['groups']):
            n_i = max(1, int(N_EVAL * g['frac']))
            print(f"  [{g['label']}] n={n_i}...")
            imgs = gen_images(source_scribble, g['prompt'], n_i, seed=SEED + i*1000)
            all_target_imgs.extend(imgs)
            target_imgs_by_group[g['label']] = imgs
        target_clip = clip_embed(all_target_imgs)

    elif mode == 'age':
        ages      = list(range(cfg['age_min'], cfg['age_max']+1, cfg['age_step']))
        n_per_age = max(1, N_EVAL // len(ages))

        age_embs = {}
        clip_model.to(device)
        for age in tqdm(ages, desc='Age target'):
            prompt = (f'a superrealistic portrait photograph of a {age}-year-old man, '
                      'studio lighting, sharp focus, photographic')
            imgs = gen_images(source_scribble, prompt, n_per_age, seed=SEED + age*7)
            tensors = torch.cat([TF.to_tensor(img).unsqueeze(0) for img in imgs]).to(device)
            with torch.no_grad():
                age_embs[age] = encode_images_clip(tensors, clip_model, clip_processor).cpu()
            target_imgs_by_group[str(age)] = imgs
        clip_model.to('cpu')
        target_clip = torch.cat([age_embs[a] for a in ages], dim=0).to(device)

    print(f'Target CLIP: {target_clip.shape}')

    # ── Evaluate each method ──
    print(f'\nEvaluating methods...')
    exp_results = {}

    for method_name, scribble in scribbles.items():
        print(f'  [{method_name}] generating {N_EVAL} images...')
        imgs = gen_images(scribble, NEUTRAL_PROMPT, N_EVAL, seed=SEED, controlnet_scale=EVAL_CONTROLNET_SCALE)
        embs = clip_embed(imgs)
        mmd  = compute_mmd(embs, target_clip).item()
        swd  = compute_swd_seeded(embs, target_clip).item()

        result = {'mmd': mmd, 'swd': swd}

        if mode in ('binary', 'multiclass'):
            sr, _ = compute_clip_softmax(
                imgs, clip_model, clip_processor, MAN_PROMPT, WOMAN_PROMPT, device)
            n_male = sum(1 for r in sr if r['label'] == 'male')
            p, lo, hi = binomial_ci(n_male, N_EVAL)
            result.update(dict(p_male=p, ci_lo=lo, ci_hi=hi,
                               n_male=n_male, n_female=N_EVAL-n_male))
            print(f'    MMD={mmd:.5f}  SWD={swd:.5f}  p(male)={p:.3f}  95%CI=[{lo:.3f},{hi:.3f}]')
        else:
            print(f'    MMD={mmd:.5f}  SWD={swd:.5f}')

        exp_results[method_name] = result
        show_image_row(imgs, f'{exp_name} — {method_name} (MMD={mmd:.4f})', n=N_PREVIEW)

    # ── Improvement vs source ──
    source_mmd = exp_results['source']['mmd']
    source_swd = exp_results['source']['swd']
    for name, r in exp_results.items():
        r['mmd_improvement']     = source_mmd - r['mmd']
        r['mmd_improvement_pct'] = (source_mmd - r['mmd']) / source_mmd * 100
        r['swd_improvement']     = source_swd - r['swd']
        r['swd_improvement_pct'] = (source_swd - r['swd']) / source_swd * 100

    all_results[exp_name] = exp_results

    # ── Results table ──
    rows = {}
    for name, r in exp_results.items():
        row = {
            'MMD':             f"{r['mmd']:.5f}",
            'Δ MMD vs source': f"{r['mmd_improvement']:+.5f}",
            'Δ MMD (%)':       f"{r['mmd_improvement_pct']:+.1f}%",
            'SWD':             f"{r['swd']:.5f}",
            'Δ SWD vs source': f"{r['swd_improvement']:+.5f}",
            'Δ SWD (%)':       f"{r['swd_improvement_pct']:+.1f}%",
        }
        if mode in ('binary', 'multiclass'):
            row['p(male)'] = f"{r['p_male']:.3f}"
            row['95% CI']  = f"[{r['ci_lo']:.3f}, {r['ci_hi']:.3f}]"
            row['n_male / n_female'] = f"{r['n_male']} / {r['n_female']}"
        rows[name] = row
    df = pd.DataFrame(rows).T
    print(f'\n{exp_name} Results:')
    display(df)

    # ── MMD / SWD bar charts ──
    names  = list(exp_results.keys())
    colors = [METHOD_COLORS.get(n, '#4C72B0') for n in names]

    for metric, source_val in (('mmd', source_mmd), ('swd', source_swd)):
        vals = [exp_results[n][metric] for n in names]
        fig, ax = plt.subplots(figsize=(8, 4))
        bars = ax.bar(names, vals, color=colors, alpha=0.85)
        ax.axhline(source_val, color='gray', linestyle='--', linewidth=1.0, label='source baseline')
        for bar, v in zip(bars, vals):
            imp = (source_val - v) / source_val * 100
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0002,
                    f'{imp:+.1f}%', ha='center', va='bottom', fontsize=9)
        ax.set_ylabel(metric.upper()); ax.set_title(f'{exp_name} — {metric.upper()} vs target (N={N_EVAL})')
        ax.legend(); ax.grid(True, axis='y', alpha=0.3)
        plt.xticks(rotation=15, ha='right')
        plt.tight_layout(); display(fig); plt.close()

    # ── p(male) CI plot (binary/multiclass only) ──
    if mode in ('binary', 'multiclass'):
        p_hats = [exp_results[n]['p_male'] for n in names]
        ci_lo  = [exp_results[n]['ci_lo']  for n in names]
        ci_hi  = [exp_results[n]['ci_hi']  for n in names]
        target_frac = cfg['groups'][0]['frac']

        fig, ax = plt.subplots(figsize=(8, 4))
        for i, (name, p, lo, hi, col) in enumerate(zip(names, p_hats, ci_lo, ci_hi, colors)):
            ax.errorbar(i, p, yerr=[[p-lo], [hi-p]], fmt='o', capsize=6,
                        linewidth=1.5, color=col, markersize=8)
        ax.axhline(target_frac, color='gray', linestyle='--', linewidth=0.8,
                   label=f'target p(male)={target_frac}')
        ax.set_xticks(range(len(names)))
        ax.set_xticklabels(names, rotation=15, ha='right')
        ax.set_ylabel('p(male)'); ax.set_title(f'{exp_name} — p(male) 95% CI (N={N_EVAL})')
        ax.set_ylim(0, 1); ax.legend(); ax.grid(True, axis='y', alpha=0.3)
        plt.tight_layout(); display(fig); plt.close()

print('\n✅ All avg_latent evaluations done.')

In [ ]:
## 6. Save Results JSON

def to_json(obj):
    if isinstance(obj, dict):  return {k: to_json(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [to_json(v) for v in obj]
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    return obj

out_path = str(_REPO / 'experiments' / 'eval_avg_latent_results.json')
with open(out_path, 'w') as f:
    json.dump(to_json(all_results), f, indent=2)
print(f'Results saved to {out_path}')

print('\n=== SUMMARY ===')
for exp_name, exp_results in all_results.items():
    print(f'\n{exp_name}:')
    for method, r in exp_results.items():
        line = f"  {method:10s}  MMD={r['mmd']:.5f} ({r['mmd_improvement_pct']:+.1f}%)  SWD={r['swd']:.5f} ({r['swd_improvement_pct']:+.1f}%)"
        if 'p_male' in r:
            line += f"  p(male)={r['p_male']:.3f}"
        print(line)